# CASMI26 v3: train-cosine top-5 + COCONUT NP fills
Peak-renormalized. CPU-only, offline. Writes `submission.csv`.

In [ ]:
"""Subformula labelling (MIST-CF lite, RDKit-free).
Assign each MS2 peak a subformula of the candidate precursor formula.
RDBE filter, ppm matching, adduct-adjusted masses.
"""
import numpy as np
from itertools import product

# monoisotopic masses
ELEM_MASS = {
    "C": 12.0, "H": 1.00782503223, "N": 14.00307400443, "O": 15.99491461957,
    "P": 30.9737619985, "S": 31.9720711744, "F": 18.99840316273,
    "Cl": 34.968852682, "Br": 78.9183376, "I": 126.9044719,
    "Na": 22.9897692809, "K": 38.9637074864,
}
ELEM_ORDER = ["C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"]

ADDUCT_DELTA = {
    "[M+H]+": 1.007276, "[M+Na]+": 22.989218, "[M+K]+": 38.963158,
    "[M+NH4]+": 18.033823, "[M-H]-": -1.007276, "[M+Cl]-": 34.968853,
    "[M+CH2O2-H]-": 44.998201, "[M+C2H4O2-H]-": 59.013851,
    "[M]+": 0.0, "[M-H2O+H]+": -17.003348, "[M-2H2O+H]+": -35.013913,
    "[2M+H]+": None, "[2M+Na]+": None, "[2M-H]-": None, "[M+2H]2+": None,
}


def parse_formula(s):
    """'C9H8N2O2' -> dict. Handles two-letter elements."""
    import re
    out = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", s):
        if el not in ELEM_MASS:
            return None
        out[el] = out.get(el, 0) + (int(n) if n else 1)
    return out


def formula_mass(f):
    return sum(ELEM_MASS[e] * n for e, n in f.items())


def rdbe(f):
    """Ring-double-bond equivalents. None if elements unsupported."""
    c = f.get("C", 0); h = f.get("H", 0); n = f.get("N", 0)
    hal = sum(f.get(e, 0) for e in ("F", "Cl", "Br", "I"))
    for e in f:
        if e not in ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"):
            return None
    return c - (h + hal) / 2 + n / 2 + 1


def enumerate_subformulae(prec_f, max_n=200000):
    """All f ⊆ prec_f with RDBE >= 0, as (counts_tuple, mass). Bounded."""
    keys = [e for e in ELEM_ORDER if e in prec_f]
    counts = [prec_f[e] for e in keys]
    # guard combinatorial explosion (e.g. C30H50...): cap by sampling coarse grid
    total = 1
    for c in counts:
        total *= (c + 1)
    subs = []
    if total <= max_n:
        for combo in product(*[range(c + 1) for c in counts]):
            if all(v == 0 for v in combo):
                continue
            f = dict(zip(keys, combo))
            if (rdbe(f) or -1) < 0:
                continue
            subs.append((combo, sum(ELEM_MASS[e] * n for e, n in zip(keys, combo))))
    else:
        # vectorized random sampling for huge combinatorial spaces
        rng = np.random.default_rng(0)
        k = len(keys)
        cm = np.array(counts)
        draws = rng.integers(0, cm + 1, size=(min(max_n * 3, 600000), k))
        draws = np.unique(draws, axis=0)
        nz = draws[np.any(draws > 0, axis=1)][:max_n]
        idx = {e: i for i, e in enumerate(keys)}
        hal_cols = [idx[e] for e in ("F", "Cl", "Br", "I") if e in idx]
        hal = nz[:, hal_cols].sum(axis=1) if hal_cols else 0
        c = nz[:, idx["C"]] if "C" in idx else 0
        h = nz[:, idx["H"]] if "H" in idx else 0
        n = nz[:, idx["N"]] if "N" in idx else 0
        ok = (c - (h + hal) / 2 + n / 2 + 1) >= 0
        sup = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")
        if any(e not in sup for e in keys):
            ok = ok & False
        nz = nz[ok][:max_n]
        mv = np.array([ELEM_MASS[e] for e in keys])
        masses = nz @ mv
        subs = [(tuple(row), float(m)) for row, m in zip(nz.tolist(), masses.tolist())]
    return keys, subs


_SUB_CACHE = {}


def subformula_masses(prec_formula_str, max_n=200000):
    """Cached (keys, masses array) for a precursor formula."""
    hit = _SUB_CACHE.get(prec_formula_str)
    if hit is not None:
        return hit
    prec_f = parse_formula(prec_formula_str)
    if prec_f is None:
        return None
    keys, subs = enumerate_subformulae(prec_f, max_n)
    arr = np.array([m for _, m in subs], dtype=float)
    _SUB_CACHE[prec_formula_str] = (keys, arr)
    return keys, arr


def label_peaks(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Greedy: for each top-N peak (by intensity), nearest subformula mass within ppm.
    Returns list of (mz, intensity, subformula_mass or None, ppm_err or None).
    Assumes fragments carry precursor adduct (MIST-CF assumption).
    """
    cached = subformula_masses(prec_formula_str)
    if cached is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    d = ADDUCT_DELTA.get(adduct)
    if d is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    mzs = np.asarray(mzs, dtype=float); intens = np.asarray(intens, dtype=float)
    order = np.argsort(-intens)[:top_n]
    _, sub_masses = cached
    out = []
    for idx in order:
        target = mzs[idx] - d  # adduct-adjusted neutral fragment mass
        if len(sub_masses) == 0:
            out.append((mzs[idx], intens[idx], None, None))
            continue
        j = int(np.argmin(np.abs(sub_masses - target)))
        err_ppm = abs(sub_masses[j] - target) / max(target, 1e-9) * 1e6
        if err_ppm <= ppm:
            out.append((mzs[idx], intens[idx], float(sub_masses[j]), float(err_ppm)))
        else:
            out.append((mzs[idx], intens[idx], None, None))
    return out


def explained_intensity(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Fraction of top-N intensity explained by subformulae. Core v1 feature."""
    labelled = label_peaks(mzs, intens, prec_formula_str, adduct, ppm, top_n)
    tot = sum(i for _, i, _, _ in labelled)
    exp = sum(i for _, i, m, _ in labelled if m is not None)
    n_hit = sum(1 for _, _, m, _ in labelled if m is not None)
    return (exp / tot if tot > 0 else 0.0), n_hit


In [ ]:
"""v2: memory-based fingerprint prediction (MIST retrieval path, no training).
Query spectra -> cosine neighbors among mass-window train spectra ->
cosine-weighted fingerprint blend -> rank candidates by Tanimoto.
"""
import numpy as np, pandas as pd
from bisect import bisect_left, bisect_right

PROJECT = "/Users/martin/Desktop/enveda-casmi26-molecule-id"


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def cosine(mz1, it1, mz2, it2, tol=0.02):
    a = np.asarray(it1, dtype=float); b = np.asarray(it2, dtype=float)
    na = float(np.sqrt((a * a).sum())); nb = float(np.sqrt((b * b).sum()))
    if na == 0 or nb == 0: return 0.0
    m1 = np.asarray(mz1, dtype=float); m2 = np.asarray(mz2, dtype=float)
    o1 = np.argsort(m1); o2 = np.argsort(m2)
    m1, a = m1[o1], a[o1]; m2, b = m2[o2], b[o2]
    i = j = 0; num = 0.0
    while i < len(m1) and j < len(m2):
        d = m1[i] - m2[j]
        if abs(d) <= tol: num += a[i] * b[j]; i += 1; j += 1
        elif d < 0: i += 1
        else: j += 1
    return num / (na * nb)


def tanimoto(a, b):
    inter = float(np.logical_and(a, b).sum())
    union = float(np.logical_or(a, b).sum())
    return inter / union if union > 0 else 0.0


def load_fp():
    df = pd.read_parquet(f"{PROJECT}/data/fingerprints.parquet")
    return {s: np.unpackbits(np.asarray(f, dtype=np.uint8)) for s, f in zip(df["smiles"], df["fp"])}


def run(n_query=30, seed=1, k_blend=15, ppm=20, min_n=200):
    rng = np.random.default_rng(seed)
    fp = load_fp()
    train = pd.read_parquet(f"{PROJECT}/data/train.parquet",
        columns=["normalized_smiles", "inchikey14", "adduct", "precursor_mz",
                 "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    groups = np.array(train["inchikey14"].unique())
    held = set(rng.choice(groups, size=min(500, len(groups) // 20), replace=False))
    qpool = train[train["inchikey14"].isin(held)]
    structs = np.array(qpool["normalized_smiles"].unique())
    rng.shuffle(structs)
    queries = list(structs[:n_query])

    db = train[~train["inchikey14"].isin(held)]
    struct = db.groupby("normalized_smiles")["neutral"].median().reset_index()
    truth = qpool.groupby("normalized_smiles")["neutral"].median().reset_index()
    struct = pd.concat([struct, truth]).drop_duplicates("normalized_smiles")
    struct = struct.sort_values("neutral").reset_index(drop=True)
    masses = struct["neutral"].values
    smi = struct["normalized_smiles"].values
    # sampled db spectra per structure for neighbor search
    db_samp = db.groupby("normalized_smiles").head(3)

    hits = {1: 0, 5: 0, 25: 0}
    rr = []
    for qi, qs in enumerate(queries):
        qspec = qpool[qpool["normalized_smiles"] == qs]
        qmass = float(np.median([neutral_mass(p, a) for p, a in zip(qspec["precursor_mz"], qspec["adduct"])]))
        tol = qmass * ppm / 1e6
        lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        pcur = ppm
        while hi - lo < min_n and pcur < 500:
            pcur *= 2; tol = qmass * pcur / 1e6
            lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        cands = list(smi[lo:hi][:2000])
        win = db_samp[db_samp["normalized_smiles"].isin(set(cands))]
        # neighbor similarities: best cosine of each db spectrum vs query spectra
        sims = []
        for _, r in qspec.iterrows():
            for _, t in win.iterrows():
                c = cosine(r["ms2_mzs"], r["ms2_normalized_intensities"],
                           t["ms2_mzs"], t["ms2_normalized_intensities"])
                if c > 0.01:
                    sims.append((c, t["normalized_smiles"]))
        sims.sort(reverse=True)
        # blend top-k distinct neighbor fingerprints
        seen, num, den = set(), None, 0.0
        for c, s in sims:
            if s in seen: continue
            seen.add(s)
            f = fp.get(s)
            if f is None: continue
            num = c * f if num is None else num + c * f
            den += c
            if len(seen) >= k_blend: break
        pred = (num / den) if den > 0 else None
        scored = []
        for s in cands:
            f = fp.get(s)
            if f is None or pred is None: t = 0.0
            else: t = tanimoto(pred > 0.3, f)
            scored.append((t, s))
        scored.sort(reverse=True)
        rank = next((i + 1 for i, (_, s) in enumerate(scored) if s == qs), 10**9)
        rr.append(1 / rank if rank <= 25 else 0.0)
        for k in hits:
            if rank <= k: hits[k] += 1
        if (qi + 1) % 10 == 0: print(f"{qi+1}/{n_query} MRR25={np.mean(rr):.3f}", flush=True)
    print(f"n={n_query} hit@1={hits[1]/n_query:.3f} hit@5={hits[5]/n_query:.3f} hit@25={hits[25]/n_query:.3f} MRR@25={np.mean(rr):.3f}")




In [ ]:
"""v3: train-cosine top-5 (floor) + COCONUT NP fills (recall lottery).
Peak renormalization (top-150) per limits analysis. RDKit-free at runtime
(fingerprints precomputed). Writes submission.csv.
"""
import numpy as np, pandas as pd, os
from bisect import bisect_left, bisect_right

_fp_cache = {}


def train_fp_lookup():
    if "tfp" not in _fp_cache:
        df = pd.read_parquet(f"{FP}/fingerprints.parquet")
        _fp_cache["tfp"] = {s: np.unpackbits(np.asarray(f, dtype=np.uint8))
                            for s, f in zip(df["normalized_smiles"] if "normalized_smiles" in df.columns else df["smiles"], df["fp"])}
    return _fp_cache["tfp"]

PROJECT = "/Users/martin/Desktop/enveda-casmi26-molecule-id"
IN = os.environ.get("CASMI_IN", f"{PROJECT}/data")
OUT = os.environ.get("CASMI_OUT", PROJECT)
FP = os.environ.get("CASMI_FP", f"{PROJECT}/data")
TOP_N = 150


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def topn(mz, it, n=TOP_N):
    mz = np.asarray(mz, dtype=float); it = np.asarray(it, dtype=float)
    if len(mz) <= n: return mz, it
    o = np.argsort(-it)[:n]
    return mz[o], it[o]


def main():
    test = pd.read_parquet(f"{IN}/test.parquet")
    test["neutral"] = [neutral_mass(p, a) for p, a in zip(test["precursor_mz"], test["adduct"])]
    mol_neutral = test.groupby("molecule_id")["neutral"].median()

    train = pd.read_parquet(f"{IN}/train.parquet",
        columns=["normalized_smiles", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    tstruct = train.groupby("normalized_smiles")["neutral"].median()
    tmass = tstruct.sort_values().values
    tsmi = tstruct.sort_values().index.values
    tsamp = train.groupby("normalized_smiles").head(2)

    coco = pd.read_parquet(f"{FP}/coconut_fp.parquet")
    cmass = coco["exact_molecular_weight"].values
    co = coco.sort_values("exact_molecular_weight").reset_index(drop=True)
    cmass = co["exact_molecular_weight"].values
    cfp = {s: np.unpackbits(np.asarray(f, dtype=np.uint8))
           for s, f in zip(co["canonical_smiles"], co["fp"])}
    cform = dict(zip(co["canonical_smiles"], co["molecular_formula"]))
    cmass_d = dict(zip(co["canonical_smiles"], co["exact_molecular_weight"]))
    from collections import Counter
    form_prior = Counter(co["molecular_formula"].tolist())
    tfp = train_fp_lookup()  # train smiles -> fp bits

    rows = []
    for mi, (mol, spectra) in enumerate(test.groupby("molecule_id")):
        qmass = float(mol_neutral.loc[mol])
        tol = qmass * 20 / 1e6
        lo = bisect_left(tmass, qmass - tol); hi = bisect_right(tmass, qmass + tol)
        pcur = 20
        while hi - lo < 200 and pcur < 500:
            pcur *= 2; tol = qmass * pcur / 1e6
            lo = bisect_left(tmass, qmass - tol); hi = bisect_right(tmass, qmass + tol)
        tcands = list(tsmi[lo:hi][:2000])
        twin = tsamp[tsamp["normalized_smiles"].isin(set(tcands))]
        qspecs = [(np.asarray(mz, dtype=float), np.asarray(it, dtype=float))
                  for mz, it in zip(spectra["ms2_mzs"], spectra["ms2_normalized_intensities"])]
        qspecs = [topn(mz, it) for mz, it in qspecs]
        tscored = []
        for s in tcands:
            best = 0.0
            for tmz, tit in zip(twin[twin["normalized_smiles"] == s]["ms2_mzs"],
                               twin[twin["normalized_smiles"] == s]["ms2_normalized_intensities"]):
                dmz, dit = topn(tmz, tit)
                for qmz, qit in qspecs:
                    c = cosine(qmz, qit, dmz, dit)
                    if c > best: best = c
            tscored.append((best, s))
        tscored.sort(reverse=True)
        top5 = [s for _, s in tscored[:5]]
        # mean fp of train top-5 as NP-analog query
        vecs = [tfp[s] for s in top5 if s in tfp]
        qfp = (np.stack(vecs).mean(axis=0) > 0.5) if vecs else None
        # COCONUT mass window
        ctol = qmass * 20 / 1e6
        clo = bisect_left(cmass, qmass - ctol); chi = bisect_right(cmass, qmass + ctol)
        cpcur = 20
        while chi - clo < 200 and cpcur < 500:
            cpcur *= 2; ctol = qmass * cpcur / 1e6
            clo = bisect_left(cmass, qmass - ctol); chi = bisect_right(cmass, qmass + ctol)
        ccands = co.iloc[clo:chi]["canonical_smiles"].tolist()[:3000]
        seen = set(top5)
        cscored = []
        for s in ccands:
            if s in seen: continue
            f = cfp.get(s)
            t = tanimoto(qfp, f) if (qfp is not None and f is not None) else 0.0
            prior = np.log1p(form_prior.get(cform.get(s, ""), 0))
            merr = abs(cmass_d[s] - qmass) / qmass
            cscored.append((t + 0.05 * prior - merr, s))
        cscored.sort(reverse=True)
        fills = [s for _, s in cscored[:20] if not (s in seen or seen.add(s))]
        # backbone: remaining train-cosine ranks fill any leftover
        rest = [s for _, s in tscored[5:] if s not in seen]
        out = (top5 + fills + rest)[:25]
        rows.append((mol, ";".join(out)))
        if (mi + 1) % 50 == 0: print(f"done {mi+1}/400", flush=True)
    pd.DataFrame(rows, columns=["molecule_id", "smiles"]).to_csv(f"{OUT}/submission.csv", index=False)
    print("wrote submission.csv")



main()
